In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from scipy.stats import mannwhitneyu
from scipy.stats import spearmanr

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
STATISTICAL_REPORT_PATH = PROJECT_ROOT / "reports" / "statistical"

STATISTICAL_REPORT_PATH.mkdir(parents=True,exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_PATH)
print("Statistical reports:", STATISTICAL_REPORT_PATH)

Project root: e:\Learning\Projects\Olist-Ecom
Processed data: e:\Learning\Projects\Olist-Ecom\data\processed
Statistical reports: e:\Learning\Projects\Olist-Ecom\reports\statistical


In [3]:
orders = pd.read_csv(PROCESSED_PATH / "orders.csv")

reviews = pd.read_csv(PROCESSED_PATH / "order_reviews.csv")

print("Orders shape:", orders.shape)
print("Reviews shape:", reviews.shape)

Orders shape: (99441, 12)
Reviews shape: (99224, 7)


In [4]:
print("Orders columns:")
print(orders.columns.tolist())

print("\nReviews columns:")
print(reviews.columns.tolist())

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'is_late', 'delivery_status']

Reviews columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [5]:
dateColumns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in dateColumns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [7]:
print(orders["is_late"].value_counts(dropna=False))

is_late
False    91614
True      7827
Name: count, dtype: int64


In [8]:
print(reviews["review_score"].value_counts(dropna=False).sort_index())

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [9]:
statData = orders[
    ["order_id","delivery_days","is_late"]
].merge(
    reviews[["order_id","review_score"]],
    on="order_id",
    how="inner"
)

print("Statistical dataset shape:", statData.shape)

statData.head()

Statistical dataset shape: (99224, 4)


,order_id,delivery_days,is_late,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,False,4
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,False,4
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,False,5
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,False,5
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,False,5


In [11]:
test1Data = statData[["is_late", "review_score"]].dropna()

print(test1Data.shape)

(99224, 2)


In [12]:
test1Data["is_late"].value_counts()

is_late
False    91523
True      7701
Name: count, dtype: int64

In [16]:
onTimeReviews = test1Data.loc[test1Data["is_late"] == False,"review_score"]

lateReviews = test1Data.loc[test1Data["is_late"] == True,"review_score"]

In [17]:
print("On-time orders:", len(onTimeReviews))
print("Late orders:", len(lateReviews))

print("\nOn-time median review:",onTimeReviews.median())

print("Late median review:",lateReviews.median())

On-time orders: 91523
Late orders: 7701

On-time median review: 5.0
Late median review: 2.0


In [18]:
mannWhitneyResult = mannwhitneyu(onTimeReviews,lateReviews,alternative="two-sided")

print(mannWhitneyResult)

MannwhitneyuResult(statistic=np.float64(538376866.5), pvalue=np.float64(0.0))


In [20]:
uStatistic = mannWhitneyResult.statistic
pValue = mannWhitneyResult.pvalue

print("U statistic:", uStatistic)
print("p-value:", pValue)

U statistic: 538376866.5
p-value: 0.0


In [21]:
n1 = len(onTimeReviews)
n2 = len(lateReviews)

rankBiserial = (2 * uStatistic / (n1 * n2)) - 1

print("Rank-biserial effect size:", rankBiserial)

Rank-biserial effect size: 0.5277032953767455


In [22]:
alpha = 0.05

if pValue < alpha:
    print(
        "Result: Reject the null hypothesis."
    )
    print(
        "There is statistically significant evidence "
        "that review-score distributions differ between "
        "late and on-time deliveries."
    )
else:
    print(
        "Result: Fail to reject the null hypothesis."
    )
    print(
        "There is not sufficient statistical evidence "
        "of a difference in review-score distributions "
        "between late and on-time deliveries."
    )

Result: Reject the null hypothesis.
There is statistically significant evidence that review-score distributions differ between late and on-time deliveries.


In [23]:
test2Data = statData[["delivery_days","review_score"]].dropna()

print("Observations:", len(test2Data))

test2Data.head()

Observations: 96359


,delivery_days,review_score
0,8.436574,4
1,13.782037,4
2,9.394213,5
3,13.208750,5
4,2.873877,5


In [24]:
spearmanResult = spearmanr(test2Data["delivery_days"],test2Data["review_score"])

print(spearmanResult)

SignificanceResult(statistic=np.float64(-0.23441220411395522), pvalue=np.float64(0.0))


In [26]:
spearmanRHO = spearmanResult.statistic
spearmanPvalue = spearmanResult.pvalue

print("Spearman rho:", spearmanRHO)
print("p-value:", spearmanPvalue)

Spearman rho: -0.23441220411395522
p-value: 0.0


In [27]:
if spearmanPvalue < 0.05:
    print(
        "There is statistically significant evidence "
        "of a monotonic association between delivery "
        "duration and review score."
    )
else:
    print(
        "There is not sufficient statistical evidence "
        "of a monotonic association between delivery "
        "duration and review score."
    )

There is statistically significant evidence of a monotonic association between delivery duration and review score.


In [28]:
if spearmanRHO > 0:
    print("The relationship is positive.")
elif spearmanRHO < 0:
    print("The relationship is negative.")
else:
    print("The relationship is approximately zero.")

The relationship is negative.


In [29]:
statisticalSummary = pd.DataFrame({
    "Test": [
        "Mann-Whitney U",
        "Spearman Rank Correlation"
    ],
    "Business Question": [
        "Do review scores differ between late and on-time deliveries?",
        "Is delivery duration associated with review score?"
    ],
    "Statistic": [uStatistic,spearmanRHO],
    "P_Value": [pValue,spearmanPvalue],
    "Alpha": [0.05,0.05]
})

statisticalSummary

,Test,Business Question,Statistic,P_Value,Alpha
0,Mann-Whitney U,Do review scores differ between late and on-ti...,5.383769e+08,0.0,0.05
1,Spearman Rank Correlation,Is delivery duration associated with review sc...,-2.344122e-01,0.0,0.05


In [32]:
rankBiserial

np.float64(0.5277032953767455)

In [33]:
test1Results = pd.DataFrame({
    "test": ["Mann-Whitney U"],
    "question": ["Do review scores differ between late and on-time deliveries?"],
    "on_time_count": [n1],
    "late_count": [n2],
    "on_time_median": [onTimeReviews.median()],
    "late_median": [lateReviews.median()],
    "u_statistic": [uStatistic],
    "p_value": [pValue],
    "rank_biserial_effect_size": [rankBiserial]
})

In [35]:
test1Results

,test,question,on_time_count,late_count,on_time_median,late_median,u_statistic,p_value,rank_biserial_effect_size
0,Mann-Whitney U,Do review scores differ between late and on-ti...,91523,7701,5.0,2.0,538376866.5,0.0,0.527703


In [36]:
test1Results.to_csv(STATISTICAL_REPORT_PATH / "test1MannWhiteny.csv", index= False)

In [37]:
test2Results = pd.DataFrame({
    "test": ["Spearman Rank Correlation"],
    "question": ["Is delivery duration associated with review score?"],
    "observations": [len(test2Data)],
    "spearman_rho": [spearmanRHO],
    "p_value": [spearmanPvalue]
})

In [38]:
test2Results.to_csv(STATISTICAL_REPORT_PATH / "test2Spearman.csv", index= False)

In [39]:
statisticalSummary.to_csv(STATISTICAL_REPORT_PATH / "StatisticalSummary.csv", index = False)

In [40]:
test1Results

,test,question,on_time_count,late_count,on_time_median,late_median,u_statistic,p_value,rank_biserial_effect_size
0,Mann-Whitney U,Do review scores differ between late and on-ti...,91523,7701,5.0,2.0,538376866.5,0.0,0.527703


In [41]:
test2Results

,test,question,observations,spearman_rho,p_value
0,Spearman Rank Correlation,Is delivery duration associated with review sc...,96359,-0.234412,0.0
